# Fine-tuned Kronos for stock indices (GPU)

1. Fine-tune Kronos-small on the daily history of 12 global indices (training data ends 2024-12-31).
2. Check it on Jan → Aug 2025; keep it only if it beats the original model there.
3. Test original vs fine-tuned on **Nifty 50** and **S&P 500**, Sep 2025 → Oct 2026, after costs.

Pass/fail rules were written down before any results; the last cell checks them.

**Run:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all` (roughly 35–50 minutes).
At the end, copy the lines starting with **`FINETUNE_RESULTS`** and **`VERDICT`** into the Claude chat.

In [ ]:
# 1. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Upstream Kronos (pinned) + this repo's autotrader code, configs and 12 index histories
import base64, io, os, sys, tarfile, subprocess
os.chdir("/content")
if not os.path.exists("Kronos"):
    subprocess.run(["git", "clone", "-q", "https://github.com/shiyu-coder/Kronos"], check=True)
os.chdir("/content/Kronos")
subprocess.run(["git", "checkout", "-q", "67b630e67f6a18c9e9be918d9b4337c960db1e9a"], check=True)
BUNDLE = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz").extractall(".")
os.makedirs("runs", exist_ok=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(sorted(os.listdir("data/universe")))

In [ ]:
# 3. Install the few missing packages (Colab already has torch, pandas, numpy)
!pip -q install einops==0.8.1 ccxt==4.5.90 prometheus-client pyyaml safetensors

In [ ]:
# 4. GPU copies of the configs (same settings, device=cuda)
import yaml
for name in ["index-daily", "index-overlay"]:
    cfg = yaml.safe_load(open(f"deploy/config/config.{name}.yaml"))
    cfg.setdefault("model", {})["device"] = "cuda"
    yaml.safe_dump(cfg, open(f"runs/gpu_{name}.yaml", "w"))
print("ok")

In [ ]:
# 5. Fine-tune (prints one line per epoch; stops early if validation stops improving)
!python -W ignore -m autotrader.finetune --data-dir data/universe --out models/kronos-small-indices --device cuda

In [ ]:
# 6. Test-window forecasts for Nifty 50 and S&P 500 with the original and the fine-tuned model
import copy, json, os, time
from autotrader.backtest import run_forecasts
from autotrader.config import load_config
from autotrader.data import load_csv
from autotrader.forecaster import KronosForecaster, NaiveBootstrapForecaster
meta = json.load(open("models/kronos-small-indices/finetune_meta.json"))
print("fine-tuning:", {k: meta[k] for k in ["improved_over_pretrained", "pretrained_val_loss", "best_val_loss", "best_epoch"]})
C = load_config("runs/gpu_index-daily.yaml")
models = {"base": KronosForecaster(C.model, "1d", calendar="weekdays")}
if meta["improved_over_pretrained"]:
    Cf = copy.deepcopy(C)
    Cf.model.predictor, Cf.model.predictor_revision = "models/kronos-small-indices", None
    models["finetuned"] = KronosForecaster(Cf.model, "1d", calendar="weekdays")
else:
    print("Fine-tuning did not beat the original model on validation data: only the base model is tested.")
naive = NaiveBootstrapForecaster(C.model, "1d")
for idx in ["NIFTY50", "SPX"]:
    df = load_csv(f"data/universe/{idx}_1d.csv")
    for tag, f in models.items():
        t = time.time()
        print(f"--- {idx} {tag}", flush=True)
        run_forecasts(C, df, f, start="2025-09-01", out=f"runs/fc_{tag}_{idx}.csv", batch=32)
        print(f"{idx} {tag}: done in {(time.time()-t)/60:.1f} min", flush=True)
    run_forecasts(C, df, naive, start="2025-09-01", out=f"runs/fc_naive_{idx}.csv", progress=False)

In [ ]:
# 7. Reports for every model x strategy x index
for tag in models:
    for strat in ["index-daily", "index-overlay"]:
        for idx in ["NIFTY50", "SPX"]:
            !python -W ignore -m autotrader.report --config runs/gpu_{strat}.yaml --data data/universe/{idx}_1d.csv \
                --capital 1000 --forecasts runs/fc_{tag}_{idx}.csv --baseline runs/fc_naive_{idx}.csv \
                --label {idx}_{tag}_{strat} --out runs/report_{idx}_{tag}_{strat}.json > /dev/null
print("reports done")

In [ ]:
# 8. Pre-registered checks. COPY THE LINES PRINTED BELOW.
import glob
def load(idx, tag, strat):
    p = f"runs/report_{idx}_{tag}_{strat}.json"
    return json.load(open(p)) if os.path.exists(p) else None
summary = {"finetune": {k: meta[k] for k in ["improved_over_pretrained", "pretrained_val_loss", "best_val_loss", "best_epoch"]}}
for idx in ["NIFTY50", "SPX"]:
    for tag in models:
        q = load(idx, tag, "index-daily")["model"]["forecast_quality"]
        g = lambda k, n: round(q[k], n) if q.get(k) is not None else None
        row = {"ic": g("information_coefficient", 3), "hit": g("directional_hit_rate", 3),
               "brier": g("brier_p_up", 3), "brier_clim": g("brier_climatology", 3),
               "spread_ratio": g("realised_to_forecast_spread", 2)}
        for strat in ["index-daily", "index-overlay"]:
            r = load(idx, tag, strat); p = r["model"]["preset"]
            tier = {t["run"]: t for t in r["model"]["fee_tiers"]}["broker 0.25%"]
            row[strat] = {k: p[k] for k in ["return", "sharpe", "max_dd", "trades", "exposure",
                                            "buy_hold_return", "buy_hold_sharpe", "buy_hold_max_dd"]}
            row[strat]["sharpe_at_0.25pct"] = tier["sharpe"]
            row[strat]["naive_sharpe"] = r["naive_baseline"]["preset"]["sharpe"]
        summary[f"{idx}_{tag}"] = row
print("FINETUNE_RESULTS", json.dumps(summary, separators=(",", ":"), default=str))

if "finetuned" not in models:
    print("VERDICT model improved: NO (fine-tuning did not beat the original model on validation data)")
else:
    def better(i):
        ft, bs = summary[f"{i}_finetuned"], summary[f"{i}_base"]
        if None in (ft["ic"], bs["ic"], ft["brier"], bs["brier"]):
            return False
        return ft["ic"] > bs["ic"] and ft["brier"] < bs["brier"]
    improved = all(better(i) for i in ["NIFTY50", "SPX"])
    def overlay_pass(i):
        o = summary[f"{i}_finetuned"]["index-overlay"]
        return (o["sharpe"] >= o["buy_hold_sharpe"] + 0.2 and o["max_dd"] > o["buy_hold_max_dd"]
                and o["sharpe"] > o["naive_sharpe"] and o["sharpe_at_0.25pct"] >= o["buy_hold_sharpe"] + 0.2)
    usable = improved and all(overlay_pass(i) for i in ["NIFTY50", "SPX"])
    print(f"VERDICT model improved: {'YES' if improved else 'NO'} | usable (overlay passes on both): {'YES' if usable else 'NO'}")

In [ ]:
# Download all outputs
import shutil
shutil.make_archive("/content/kronos_finetune_runs", "zip", "runs")
try:
    from google.colab import files; files.download("/content/kronos_finetune_runs.zip")
except Exception as e:
    print("download manually from the Files panel:", e)